# CoRe-TFM: bounded, resumable Q1-readiness benchmark

This Colab notebook runs a leakage-free five-fold benchmark for CoRe-TFM. The primary model family contains **TabICLv2 and TabPFN-3**; **CatBoost** is a non-TFM boundary baseline. To make the complete protocol feasible in ordinary Colab sessions, every outer fold uses a deterministic, class-coverage-preserving maximum of 256 training rows and 128 test rows.

This is a fresh bounded-context run: it never mixes earlier full-context or TabFM folds. It fixes missing-target, Diamonds-target, and rare-class split failures; checkpoints every fold; caches each dataset once; and can be safely resumed after a Colab timeout.

**How to run:** select **Runtime → Run all** on the first launch. After a reconnect, rerun Cells 2–11 to restore Python state, then rerun only the incomplete 12A–12E shard. Google Drive checkpoints prevent completed folds from repeating.


In [1]:
# Local NVIDIA/CUDA setup replacing the Colab-only Cell 2.
FULL_Q1_RUN = True
RUN_ID = 'multi_seed/seed_42_train_256'
PROTOCOL_REVISION = 'rtx3050_cuda_seed42_train256_test128'

from pathlib import Path
import importlib.metadata
import json
import os
import platform
import subprocess
import sys

ROOT = Path('D:\\core-tfm').resolve()
DRIVE_BASE = Path('D:\\core-tfm\\results\\core_tfm_submission_full_v1').resolve()
RUN = DRIVE_BASE / RUN_ID
MODEL_CACHE = DRIVE_BASE / "model_cache"
RUN.mkdir(parents=True, exist_ok=True)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE / "huggingface")
os.environ["TABPFN_MODEL_CACHE_DIR"] = str(MODEL_CACHE / "tabpfn")
os.environ["TABPFN_NO_BROWSER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "8")

FREEZE_PATH = RUN / "frozen_source_revisions.json"

def run(command, *, cwd=None):
    print("RUN:", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if FREEZE_PATH.exists():
    frozen = json.loads(FREEZE_PATH.read_text(encoding="utf-8"))
    if frozen.get("core_tfm_commit") != head:
        raise RuntimeError(
            f"Run is frozen to {frozen.get('core_tfm_commit')} but repository HEAD is {head}. "
            "Checkout the frozen commit or use a new run directory."
        )
else:
    frozen = {"core_tfm_commit": head}

frozen.update({
    "protocol_revision": PROTOCOL_REVISION,
    "execution_platform": "windows_local_nvidia_cuda",
    "bounded_context_protocol": {"max_train_rows": 256, "max_test_rows": 128},
    "hardware_target": "RTX 3050 Laptop 4GB",
})
FREEZE_PATH.write_text(json.dumps(frozen, indent=2), encoding="utf-8")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(ROOT)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not visible to PyTorch. Run the suite preflight first.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3))
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

tabpfn_token = os.environ.get("TABPFN_TOKEN", "").strip()
if not tabpfn_token:
    raise RuntimeError(
        "TABPFN_TOKEN is not set in this PowerShell session. Set $env:TABPFN_TOKEN before running."
    )
os.environ["TABPFN_TOKEN"] = tabpfn_token

# Headless license/token validation when the installed TabPFN exposes this API.
try:
    from tabpfn.browser_auth import ensure_license_accepted, verify_token
    from tabpfn.errors import TabPFNLicenseError
    from tabpfn.settings import settings
    token_status = verify_token(tabpfn_token, settings.tabpfn.auth_api_url)
    if token_status is not True:
        raise RuntimeError("TABPFN_TOKEN is invalid or the Prior Labs server is unreachable.")
    try:
        ensure_license_accepted(hf_repo_id="tabpfn_3")
    except TabPFNLicenseError as exc:
        if "browser login is disabled" in str(exc):
            raise RuntimeError("Accept the TabPFN-3 license in the Prior Labs account and rerun.") from exc
        raise
except ImportError:
    pass

ENV_PATH = RUN / "environment_metadata.json"
env_payload = {
    "source_commit": head,
    "platform": platform.platform(),
    "python": sys.version,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "vram_bytes": int(torch.cuda.get_device_properties(0).total_memory),
    "seed": 42,
    "train_limit": 256,
    "test_limit": 128,
}
ENV_PATH.write_text(json.dumps(env_payload, indent=2), encoding="utf-8")
print("Persistent run directory:", RUN)
print("Protocol revision:", PROTOCOL_REVISION)


GPU: NVIDIA GeForce RTX 3050 Laptop GPU
PyTorch: 2.11.0+cu128 CUDA runtime: 12.8
VRAM GiB: 4.0


D:\core-tfm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Persistent run directory: D:\core-tfm\results\core_tfm_submission_full_v1\multi_seed\seed_42_train_256
Protocol revision: rtx3050_cuda_seed42_train256_test128


In [2]:
# Cell 3 — fixed amended protocol. Completed eligible folds persist across reconnects.
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import shutil
import sys
import time
import traceback
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.model_selection import StratifiedKFold, train_test_split

EXECUTION_PROFILE = "q1_full" if FULL_Q1_RUN else "smoke"
DATASETS = [
    "anneal", "car", "credit", "customer", "marketing",
    "mic", "nursery", "phishing", "wine", "diamonds",
] if FULL_Q1_RUN else ["car", "wine"]

TFM_MODELS = ["tabiclv2", "tabpfn3"]
BOUNDARY_MODELS = ["catboost"] if FULL_Q1_RUN else []
MODELS = TFM_MODELS + BOUNDARY_MODELS if FULL_Q1_RUN else ["tabiclv2"]
EXCLUDED_MODELS = ["tabfm"]

OUTER_FOLDS = 5
INNER_VALIDATION_FRACTION = 0.20
SENSITIVITY_DATASETS = set(DATASETS)
SENSITIVITY_FRACTIONS = (0.25, 0.50, 1.00)  # fractions of the fitted validation pool
SEED = 42
WEIGHTS = (0.0, 0.25, 0.5, 0.75, 1.0)
PENALTIES = (0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0)
TABICL_N_ESTIMATORS = 2 if FULL_Q1_RUN else 1
CATBOOST_ITERATIONS = 100
CATBOOST_DEPTH = 6
CATBOOST_LEARNING_RATE = 0.05
SESSION_TIME_BUDGET_MINUTES = 120
MAX_TRAIN_ROWS = 256
MAX_TEST_ROWS = 128

RUN_CONTROLLED_REPLICATIONS = False
RUN_SELECTION_ABLATIONS = FULL_Q1_RUN
RUN_VALIDATION_SENSITIVITY = FULL_Q1_RUN
BUILD_MANUSCRIPTS = False

assert 0 < INNER_VALIDATION_FRACTION < 0.5
assert CATBOOST_ITERATIONS > 0
assert set(TFM_MODELS) == {"tabiclv2", "tabpfn3"}
assert not (set(MODELS) & set(EXCLUDED_MODELS))

protocol = {
    "run_id": RUN_ID,
    "protocol_revision": PROTOCOL_REVISION,
    "profile": EXECUTION_PROFILE,
    "datasets": DATASETS,
    "models": MODELS,
    "tfm_models_for_primary_claim": TFM_MODELS,
    "boundary_models_for_secondary_analysis": BOUNDARY_MODELS,
    "excluded_models": EXCLUDED_MODELS,
    "outer_folds": OUTER_FOLDS,
    "inner_validation_fraction": INNER_VALIDATION_FRACTION,
    "sensitivity_datasets": sorted(SENSITIVITY_DATASETS),
    "sensitivity_fractions": SENSITIVITY_FRACTIONS,
    "seed": SEED,
    "weights": WEIGHTS,
    "penalties": PENALTIES,
    "tabicl_n_estimators": TABICL_N_ESTIMATORS,
    "max_train_rows_per_outer_fold": MAX_TRAIN_ROWS,
    "max_test_rows_per_outer_fold": MAX_TEST_ROWS,
    "sampling": "deterministic; training target-class coverage guaranteed",
    "sensitivity_design": "nested label subsets of one fitted validation pool; no refitting",
    "catboost": {
        "version": "1.2.10", "iterations": CATBOOST_ITERATIONS,
        "depth": CATBOOST_DEPTH, "learning_rate": CATBOOST_LEARNING_RATE,
    },
    "session_time_budget_minutes": SESSION_TIME_BUDGET_MINUTES,
    "amendment_is_outcome_blind": True,
}
(RUN / "protocol.json").write_text(json.dumps(protocol, indent=2), encoding="utf-8")
(RUN / "protocol_amendment.json").write_text(json.dumps({
    "date": "2026-08-23",
    "change": "Bound every fold to at most 256 training and 128 test rows; use two TabICL estimators; reuse fitted validation predictions for sensitivity.",
    "reason": "Make the complete five-fold, ten-dataset, three-model protocol feasible in standard Colab sessions.",
    "outcome_scores_used_to_choose_replacement": False,
    "claim_guard": "Claims apply to the bounded-context protocol; CatBoost is not a TFM and is excluded from the cross-TFM primary claim.",
}, indent=2), encoding="utf-8")
print("Run directory:", RUN)
print("Primary TFMs:", TFM_MODELS, "| secondary boundary baseline:", BOUNDARY_MODELS)


Run directory: D:\core-tfm\results\core_tfm_submission_full_v1\multi_seed\seed_42_train_256
Primary TFMs: ['tabiclv2', 'tabpfn3'] | secondary boundary baseline: ['catboost']


In [3]:
# Cell 4 — repository tests and guarded model factories.
import gc
import torch
from catboost import CatBoostClassifier
from pandas.api.types import is_bool_dtype, is_numeric_dtype

test_env = {**os.environ, "PYTHONPATH": str(SRC)}
subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], cwd=ROOT, env=test_env, check=True)
subprocess.run(
    [sys.executable, "experiments/smoke_synthetic.py"],
    cwd=ROOT, env=test_env, check=True,
)

from core_tfm.models.base import ProbabilisticClassifierAdapter
from core_tfm.models.tfm_adapters import tabiclv2_adapter, tabpfn3_adapter


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def streaming_sha256(path, chunk_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


class CatBoostFrameAdapter(ProbabilisticClassifierAdapter):
    """Deterministic CatBoost adapter with fold-local, leakage-free cleaning."""

    def __init__(self, *, iterations, depth, learning_rate, seed, thread_count=2):
        self._params = dict(
            iterations=iterations,
            depth=depth,
            learning_rate=learning_rate,
            random_seed=seed,
            thread_count=thread_count,
            loss_function="MultiClass",
            allow_writing_files=False,
            verbose=False,
        )

    def _transform(self, X):
        frame = X.loc[:, self._columns].copy()
        for column in self._categorical_columns:
            frame[column] = (
                frame[column].astype("string").fillna("__MISSING__").astype(str)
            )
        for column in self._numeric_columns:
            values = pd.to_numeric(frame[column], errors="coerce").replace(
                [np.inf, -np.inf], np.nan
            )
            frame[column] = values.fillna(self._numeric_fill[column]).astype(float)
        return frame

    def fit(self, X, y):
        self._columns = list(X.columns)
        self._categorical_columns = [
            column for column in self._columns
            if is_bool_dtype(X[column].dtype) or not is_numeric_dtype(X[column].dtype)
        ]
        self._numeric_columns = [
            column for column in self._columns if column not in self._categorical_columns
        ]
        self._numeric_fill = {}
        for column in self._numeric_columns:
            values = pd.to_numeric(X[column], errors="coerce").replace(
                [np.inf, -np.inf], np.nan
            )
            median = values.median()
            self._numeric_fill[column] = 0.0 if pd.isna(median) else float(median)
        self._model = CatBoostClassifier(**self._params)
        self._model.fit(
            self._transform(X), np.asarray(y),
            cat_features=self._categorical_columns,
        )
        return self

    def predict_proba(self, X):
        return np.asarray(self._model.predict_proba(self._transform(X)), dtype=np.float64)

    @property
    def classes_(self):
        return np.asarray(self._model.classes_)


def catboost_adapter(iterations):
    return CatBoostFrameAdapter(
        iterations=iterations,
        depth=CATBOOST_DEPTH,
        learning_rate=CATBOOST_LEARNING_RATE,
        seed=SEED,
        thread_count=2,
    )


def factory_for(name):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if name == "tabiclv2":
        return lambda: tabiclv2_adapter(
            device=device, n_estimators=TABICL_N_ESTIMATORS, kv_cache=False,
            random_state=SEED, n_jobs=2, verbose=False,
        )
    if name == "tabpfn3":
        return lambda: tabpfn3_adapter(device=device, random_state=SEED)
    if name == "catboost":
        return lambda: catboost_adapter(CATBOOST_ITERATIONS)
    raise KeyError(name)


def preflight_factory_for(name):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if name == "tabiclv2":
        return lambda: tabiclv2_adapter(
            device=device, n_estimators=1, kv_cache=False,
            random_state=SEED, n_jobs=2, verbose=False,
        )
    if name == "tabpfn3":
        return lambda: tabpfn3_adapter(device=device, random_state=SEED)
    if name == "catboost":
        return lambda: catboost_adapter(30)
    raise KeyError(name)


print("Factories configured:", MODELS)


Factories configured: ['tabiclv2', 'tabpfn3', 'catboost']


In [4]:
# Cell 5 — dataset loading with authoritative Wine and Diamonds reconstruction.
import io
import urllib.request
import zipfile
from sklearn.datasets import fetch_openml
from core_tfm.data.openml import load_pair_dataset, PairDataset

UCI_WINE_URL = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"
DIAMONDS_OPENML_ID = 42225


def sha256(data):
    return hashlib.sha256(data).hexdigest()


def load_wine_canonical():
    with urllib.request.urlopen(UCI_WINE_URL, timeout=90) as response:
        archive = response.read()
    with zipfile.ZipFile(io.BytesIO(archive)) as zf:
        red_bytes = zf.read("winequality-red.csv")
        white_bytes = zf.read("winequality-white.csv")
    red = pd.read_csv(io.BytesIO(red_bytes), sep=";")
    white = pd.read_csv(io.BytesIO(white_bytes), sep=";")
    assert (len(red), len(white)) == (1599, 4898)
    a = pd.concat([
        red.pop("quality").astype(int),
        white.pop("quality").astype(int) - 2,
    ], ignore_index=True).astype("category")
    b = pd.Series(["red"] * len(red) + ["white"] * len(white), dtype="category")
    provenance = {
        "route": "canonical UCI reconstruction",
        "url": UCI_WINE_URL,
        "archive_sha256": sha256(archive),
        "red_csv_sha256": sha256(red_bytes),
        "white_csv_sha256": sha256(white_bytes),
    }
    return PairDataset(
        "wine", pd.concat([red, white], ignore_index=True), a, b
    ), provenance


def load_diamonds_pair():
    # OpenML 42225 declares continuous price as its default supervised target.
    # CoRe-TFM instead needs the pre-specified categorical pair (cut, color).
    # Reconstruct from the full frame so price remains an ordinary predictor.
    dataset = fetch_openml(
        data_id=DIAMONDS_OPENML_ID, as_frame=True, parser="auto"
    )
    frame = dataset.frame.copy().reset_index(drop=True)
    expected = {"cut", "color", "price"}
    missing = expected - set(frame.columns)
    if missing:
        raise KeyError(f"Diamonds reconstruction missing columns: {sorted(missing)}")
    a = frame.pop("cut").astype("category")
    b = frame.pop("color").astype("category")
    assert len(frame) == 53_940
    assert a.nunique() == 5 and b.nunique() == 7
    fingerprint_frame = frame.copy()
    fingerprint_frame["__a_cut__"] = a.astype(str)
    fingerprint_frame["__b_color__"] = b.astype(str)
    raw = fingerprint_frame.to_csv(index=False).encode()
    provenance = {
        "route": "OpenML full-frame categorical-pair reconstruction",
        "openml_id": DIAMONDS_OPENML_ID,
        "declared_openml_target": str(dataset.target_names),
        "target_a": "cut",
        "target_b": "color",
        "price_role": "predictor",
        "post_reconstruction_sha256": sha256(raw),
    }
    return PairDataset("diamonds", frame, a, b), provenance


def load_dataset(name):
    if name == "wine":
        return load_wine_canonical()
    if name == "diamonds":
        return load_diamonds_pair()
    ds = load_pair_dataset(name)
    frame = ds.X.copy()
    frame["__a__"] = ds.a.astype(str)
    frame["__b__"] = ds.b.astype(str)
    raw = frame.to_csv(index=False).encode()
    return ds, {
        "route": "OpenML through repository loader",
        "dataset": name,
        "post_preprocessing_sha256": sha256(raw),
    }


In [5]:
# Cell 6 — fold-level evaluator. Primary, ablation, and sensitivity evidence
# share fitted views so the notebook never repeats the full matrix unnecessarily.
from core_tfm.inference.extract import extract_pair_predictions
from core_tfm.metrics.distributions import total_variation, marginal_distortion
from core_tfm.metrics.scoring import (
    joint_brier, joint_log_loss, joint_expected_calibration_error,
    conditional_log_losses,
)
from core_tfm.reconciliation.baselines import arithmetic_pool, geometric_pool, independent_joint
from core_tfm.reconciliation.mpr import marginal_preserving_reconciliation
from core_tfm.reconciliation.soft import soft_reconciliation
from core_tfm.reconciliation.selective import select_reconciliation_policy, apply_reconciliation_policy


def clean_target_mask(series):
    text = series.astype("string").str.strip().str.lower()
    return series.notna() & ~text.isin({"", "nan"})


def stratifier(a, b, n_splits):
    # Joint labels are preferred even when rare joint cells have fewer than
    # n_splits observations. Training-class coverage is verified separately.
    joint = (a.astype(str) + "||" + b.astype(str)).to_numpy()
    return joint, "joint(A,B), coverage-validated"


def validated_outer_splits(X, a, b, n_splits):
    candidates = [
        ((a.astype(str) + "||" + b.astype(str)).to_numpy(), "joint(A,B)"),
        (a.astype(str).to_numpy(), "A"),
        (b.astype(str).to_numpy(), "B"),
    ]
    classes_a, classes_b = set(a.astype(str)), set(b.astype(str))
    for labels, route in candidates:
        for offset in range(50):
            splitter = StratifiedKFold(
                n_splits=n_splits, shuffle=True, random_state=SEED + offset
            )
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", UserWarning)
                splits = list(splitter.split(X, labels))
            if all(
                set(a.iloc[train].astype(str)) == classes_a
                and set(b.iloc[train].astype(str)) == classes_b
                for train, _ in splits
            ):
                return splits, f"{route}; seed={SEED + offset}; train-class coverage checked"
    raise ValueError(
        "Could not construct five outer folds whose training sets contain every A/B class."
    )


def capped_indices(indices, a, b, limit, seed, require_target_coverage):
    """Deterministically cap a partition without losing any target class."""
    indices = np.asarray(indices, dtype=int)
    if limit is None or len(indices) <= limit:
        return np.sort(indices)
    rng = np.random.default_rng(seed)
    mandatory = set()
    if require_target_coverage:
        for target in (a, b):
            for value in pd.unique(target.iloc[indices].astype(str)):
                candidates = indices[target.iloc[indices].astype(str).to_numpy() == value]
                mandatory.add(int(rng.choice(candidates)))
    if len(mandatory) > limit:
        raise ValueError(
            f"Row cap {limit} is smaller than the required class-coverage set."
        )
    remaining = np.array([index for index in indices if index not in mandatory], dtype=int)
    needed = limit - len(mandatory)
    selected = list(mandatory)
    if needed:
        selected.extend(rng.choice(remaining, size=needed, replace=False).tolist())
    return np.sort(np.asarray(selected, dtype=int))

def extract(factory, X, a, b, train_idx, eval_idx):
    return extract_pair_predictions(
        factory,
        X.iloc[train_idx].reset_index(drop=True),
        a.iloc[train_idx].reset_index(drop=True),
        b.iloc[train_idx].reset_index(drop=True),
        X.iloc[eval_idx].reset_index(drop=True),
        a_test=a.iloc[eval_idx].reset_index(drop=True),
        b_test=b.iloc[eval_idx].reset_index(drop=True),
    )


def prepared_dataset(dataset_name):
    ds, provenance = load_dataset(dataset_name)
    X = ds.X.copy().reset_index(drop=True)
    a = ds.a.reset_index(drop=True)
    b = ds.b.reset_index(drop=True)

    valid = clean_target_mask(a) & clean_target_mask(b)
    removed = int((~valid).sum())
    X, a, b = X.loc[valid].reset_index(drop=True), a.loc[valid].reset_index(drop=True), b.loc[valid].reset_index(drop=True)
    if isinstance(a.dtype, pd.CategoricalDtype):
        a = a.cat.remove_unused_categories()
    if isinstance(b.dtype, pd.CategoricalDtype):
        b = b.cat.remove_unused_categories()
    if min(a.astype(str).value_counts().min(), b.astype(str).value_counts().min()) < 2:
        raise ValueError("A target contains a singleton class and cannot support held-out evaluation.")

    for column in X.columns:
        if X[column].dtype == object:
            X[column] = X[column].astype("category")
    provenance = dict(provenance)
    provenance.update({
        "target_rows_removed_by_notebook": removed,
        "post_clean_rows": len(X),
        "target_cleaning": "drop true missing, empty-string, and literal 'nan' targets",
    })
    return X, a, b, provenance


def ablation_joint(spec, j1, j2, p_a, p_b):
    kind, weight, penalty = spec
    if kind == "j1": return j1
    if kind == "j2": return j2
    if kind == "arithmetic": return arithmetic_pool(j1, j2)
    if kind == "geometric": return geometric_pool(j1, j2, weight=weight)
    if kind == "hard":
        return marginal_preserving_reconciliation(
            j1, j2, p_a, p_b, reference_weight=weight
        ).joint
    if kind == "soft":
        return soft_reconciliation(
            j1, j2, p_a, p_b, reference_weight=weight,
            lambda_a=penalty, lambda_b=penalty,
        ).joint
    raise ValueError(kind)


ABLATION_FAMILIES = {
    "raw_only": [("j1", 0.5, None), ("j2", 0.5, None)],
    "pool_only": [("arithmetic", 0.5, None)]
        + [("geometric", weight, None) for weight in WEIGHTS],
    "repair_only": [("hard", weight, None) for weight in WEIGHTS]
        + [("soft", weight, penalty) for weight in WEIGHTS for penalty in PENALTIES],
}
ABLATION_FAMILIES["full"] = sum(ABLATION_FAMILIES.values(), [])


def best_ablation_spec(specs, view, y_a, y_b):
    losses = [
        joint_log_loss(
            ablation_joint(
                spec, view.j_b_then_a, view.j_a_then_b, view.p_a, view.p_b
            ),
            y_a,
            y_b,
        )
        for spec in specs
    ]
    return specs[int(np.argmin(losses))]


def inner_split(outer_train, a, b, fraction, fold):
    local_a = a.iloc[outer_train].reset_index(drop=True)
    local_b = b.iloc[outer_train].reset_index(drop=True)
    candidates = [
        (local_a.astype(str) + "||" + local_b.astype(str)).to_numpy(),
        local_a.astype(str).to_numpy(),
        local_b.astype(str).to_numpy(),
        None,
    ]
    required_a, required_b = set(local_a.astype(str)), set(local_b.astype(str))
    for attempt in range(100):
        labels = candidates[attempt % len(candidates)]
        if labels is not None and pd.Series(labels).value_counts().min() < 2:
            labels = None
        try:
            train, validation = train_test_split(
                outer_train,
                test_size=fraction,
                random_state=10_000 + fold + attempt,
                stratify=labels,
            )
        except ValueError:
            continue
        if (
            set(a.iloc[train].astype(str)) == required_a
            and set(b.iloc[train].astype(str)) == required_b
        ):
            return np.asarray(train), np.asarray(validation)
    raise ValueError("Could not create an inner split with complete A/B training-class coverage.")


def evaluate_fold(dataset_name, model_name, fold):
    """Evaluate exactly one outer fold and return all reusable evidence rows."""
    X, a, b, provenance = prepared_dataset(dataset_name)
    splits, stratification = validated_outer_splits(
        X, a, b, OUTER_FOLDS
    )
    outer_train_full, test_idx_full = splits[fold - 1]
    dataset_seed = sum((index + 1) * ord(char) for index, char in enumerate(dataset_name))
    outer_train = capped_indices(
        outer_train_full, a, b, MAX_TRAIN_ROWS,
        SEED + 10_000 * fold + dataset_seed, True,
    )
    test_idx = capped_indices(
        test_idx_full, a, b, MAX_TEST_ROWS,
        SEED + 20_000 * fold + dataset_seed, False,
    )
    factory = factory_for(model_name)

    inner_train, val_idx = inner_split(
        outer_train, a, b, INNER_VALIDATION_FRACTION, fold
    )
    val = extract(factory, X, a, b, inner_train, val_idx)
    test = extract(factory, X, a, b, outer_train, test_idx)
    vp, p = val.predictions, test.predictions

    selection = select_reconciliation_policy(
        vp.j_b_then_a, vp.j_a_then_b, vp.p_a, vp.p_b,
        val.y_a_encoded, val.y_b_encoded,
        weights=WEIGHTS, marginal_penalties=PENALTIES,
    )
    methods = {
        "j1_b_then_a": p.j_b_then_a,
        "j2_a_then_b": p.j_a_then_b,
        "independent": independent_joint(p.p_a, p.p_b),
        "arithmetic": arithmetic_pool(p.j_b_then_a, p.j_a_then_b),
        "geometric": geometric_pool(p.j_b_then_a, p.j_a_then_b),
        "hard_core": marginal_preserving_reconciliation(
            p.j_b_then_a, p.j_a_then_b, p.p_a, p.p_b
        ).joint,
        "soft_core_lambda_1": soft_reconciliation(
            p.j_b_then_a, p.j_a_then_b, p.p_a, p.p_b,
            lambda_a=1, lambda_b=1,
        ).joint,
        "selective_core": apply_reconciliation_policy(
            selection, p.j_b_then_a, p.j_a_then_b, p.p_a, p.p_b
        ),
    }

    primary_rows = []
    for method, q in methods.items():
        assert np.isfinite(q).all() and np.allclose(q.sum((1, 2)), 1, atol=1e-5)
        c_a, c_b = conditional_log_losses(q, test.y_a_encoded, test.y_b_encoded)
        primary_rows.append({
            "dataset": dataset_name, "model": model_name, "fold": fold,
            "method": method,
            "joint_nll": joint_log_loss(q, test.y_a_encoded, test.y_b_encoded),
            "joint_brier": joint_brier(q, test.y_a_encoded, test.y_b_encoded),
            "joint_ece_15": joint_expected_calibration_error(
                q, test.y_a_encoded, test.y_b_encoded
            ),
            "conditional_nll_a_given_b": c_a,
            "conditional_nll_b_given_a": c_b,
            "marginal_distortion": float(
                marginal_distortion(q, p.p_a, p.p_b).mean()
            ),
            "factorization_tv": float(
                total_variation(p.j_b_then_a, p.j_a_then_b).mean()
            ),
            "selected_policy": selection.policy.name,
            "selected_weight": selection.policy.weight,
            "selected_lambda": selection.policy.marginal_penalty,
            "n_train": len(outer_train), "n_validation": len(val_idx),
            "n_test": len(test_idx), "stratification": stratification,
        })

    ablation_rows = []
    for family, specs in ABLATION_FAMILIES.items():
        chosen = best_ablation_spec(
            specs, vp, val.y_a_encoded, val.y_b_encoded
        )
        q = ablation_joint(
            chosen, p.j_b_then_a, p.j_a_then_b, p.p_a, p.p_b
        )
        ablation_rows.append({
            "dataset": dataset_name, "model": model_name, "fold": fold,
            "family": family, "selected_kind": chosen[0],
            "selected_weight": chosen[1], "selected_penalty": chosen[2],
            "joint_nll": joint_log_loss(q, test.y_a_encoded, test.y_b_encoded),
            "marginal_distortion": float(
                marginal_distortion(q, p.p_a, p.p_b).mean()
            ),
        })

    sensitivity_rows = []
    if RUN_VALIDATION_SENSITIVITY and dataset_name in SENSITIVITY_DATASETS:
        # Reuse the already-fitted validation views. Nested label subsets isolate
        # policy-selection sample size without performing any additional model fits.
        order = np.random.default_rng(
            SEED + 30_000 * fold + dataset_seed
        ).permutation(len(val_idx))
        for fraction in SENSITIVITY_FRACTIONS:
            n_selection = max(8, int(np.ceil(len(order) * fraction)))
            subset = np.sort(order[:min(n_selection, len(order))])
            selected = select_reconciliation_policy(
                vp.j_b_then_a[subset], vp.j_a_then_b[subset],
                vp.p_a[subset], vp.p_b[subset],
                val.y_a_encoded[subset], val.y_b_encoded[subset],
                weights=WEIGHTS, marginal_penalties=PENALTIES,
            )
            q = apply_reconciliation_policy(
                selected, p.j_b_then_a, p.j_a_then_b, p.p_a, p.p_b
            )
            sensitivity_rows.append({
                "dataset": dataset_name, "model": model_name, "fold": fold,
                "validation_fraction": fraction,
                "validation_sample_size": int(len(subset)),
                "joint_nll": joint_log_loss(
                    q, test.y_a_encoded, test.y_b_encoded
                ),
                "selected_policy": selected.policy.name,
                "selected_weight": selected.policy.weight,
                "selected_lambda": selected.policy.marginal_penalty,
            })

    manifest = {
        "dataset": dataset_name, "model": model_name, "fold": fold,
        "n_rows": len(X), "n_features": X.shape[1],
        "n_outer_train_available": len(outer_train_full),
        "n_outer_train_used": len(outer_train),
        "n_outer_test_available": len(test_idx_full),
        "n_outer_test_used": len(test_idx),
        "provenance": provenance, "seed": SEED,
    }
    return primary_rows, ablation_rows, sensitivity_rows, manifest


## Final evidence plan and amended scope

The active matrix is ten datasets × two released TFMs (TabICLv2 and TabPFN-3) plus one strong classical boundary baseline (CatBoost) × five outer folds. Every fold uses at most 256 training and 128 test rows, selected deterministically while preserving every A/B class in training. CatBoost is included to test model-family generality, but it is not described or counted as a TFM.

TabFM is excluded for operational incompatibility and impractical Colab runtime. This amendment is recorded before using comparative outcome scores. Existing TabICLv2 and TabPFN-3 folds are retained; TabFM rows are archived and excluded from all active gates and analyses.

The primary estimand remains Selective CoRe minus arithmetic pooling on joint NLL, averaged across the two TFMs within each dataset and tested over ten dataset blocks. CatBoost is secondary. All folds use inner-validation policy choice and untouched outer-test evaluation. Claims are explicitly limited to this bounded-context regime.


In [6]:
# Cell 8 — fast one-estimator model preflight. It does not use final ensemble sizes.
toy_x = pd.DataFrame({
    "n": np.arange(12, dtype=float),
    "c": pd.Categorical(["x", "y"] * 6),
})
toy_y = pd.Series(["a", "b"] * 6)
full_preflight = {}

for model_name in MODELS:
    started = time.time()
    print(f"START lightweight preflight: {model_name}", flush=True)
    try:
        fitted = preflight_factory_for(model_name)().fit(toy_x, toy_y)
        proba = np.asarray(fitted.predict_proba(toy_x.iloc[:2]))
        assert proba.shape == (2, 2)
        assert np.isfinite(proba).all()
        assert np.allclose(proba.sum(axis=1), 1, atol=1e-5)
        full_preflight[model_name] = {
            "ok": True, "seconds": time.time() - started
        }
    except Exception as exc:
        full_preflight[model_name] = {
            "ok": False, "seconds": time.time() - started,
            "error": repr(exc), "traceback": traceback.format_exc(),
        }
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

(RUN / "full_model_preflight.json").write_text(
    json.dumps(full_preflight, indent=2), encoding="utf-8"
)

print(json.dumps(full_preflight, indent=2))
assert all(result["ok"] for result in full_preflight.values())

START lightweight preflight: tabiclv2


D:\core-tfm\.venv\Lib\site-packages\torch\nn\modules\module.py:1370: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(


START lightweight preflight: tabpfn3


START lightweight preflight: catboost


{
  "tabiclv2": {
    "ok": true,
    "seconds": 2.923746109008789
  },
  "tabpfn3": {
    "ok": true,
    "seconds": 5.1120147705078125
  },
  "catboost": {
    "ok": true,
    "seconds": 0.3357834815979004
  }
}


## Full-run protocol: claims, licenses, and stopping rules

This amended profile runs **10 dataset pairs × (2 frozen TFMs + 1 CatBoost boundary baseline) × 5 outer folds**, with at most 256 training and 128 test rows per fold. It additionally runs controlled studies, candidate-family ablations, and validation-sample-size sensitivity analysis using nested subsets of already-fitted validation predictions.

TabPFN-3 requires license acceptance and a token. CatBoost is pinned to version 1.2.10 and runs on CPU with fixed hyperparameters. The paper may claim cross-TFM behavior only from TabICLv2 and TabPFN-3 and only for the bounded-context protocol. CatBoost supplies secondary model-family evidence.

The pre-specified comparator remains arithmetic pooling. No significance claim is permitted if the final evidence gate fails.


In [7]:
# Cell 10 — mandatory hardware and dependency provenance.
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

hardware = {
    "python": sys.version,
    "platform": platform.platform(),
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_version": torch.version.cuda,
    "torch": version("torch"),
    "tabicl": version("tabicl"),
    "tabpfn": version("tabpfn"),
    "catboost": version("catboost"),
    "active_models": MODELS,
    "primary_tfm_models": TFM_MODELS,
    "boundary_models": BOUNDARY_MODELS,
    "frozen_revisions": json.loads(FREEZE_PATH.read_text(encoding="utf-8")),
}
if hardware["cuda_available"]:
    hardware["gpu_names"] = [
        torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())
    ]
    try:
        hardware["nvidia_smi"] = subprocess.check_output([
            "nvidia-smi", "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader",
        ], text=True).strip()
    except Exception as exc:
        hardware["nvidia_smi_error"] = repr(exc)
(RUN / "hardware_and_model_environment.json").write_text(
    json.dumps(hardware, indent=2), encoding="utf-8"
)
print(json.dumps(hardware, indent=2))
assert hardware["catboost"] == "1.2.10"


{
  "python": "3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]",
  "platform": "Windows-10-10.0.26200-SP0",
  "cuda_available": true,
  "cuda_version": "12.8",
  "torch": "2.11.0+cu128",
  "tabicl": "2.1.1",
  "tabpfn": "8.5.0",
  "catboost": "1.2.10",
  "active_models": [
    "tabiclv2",
    "tabpfn3",
    "catboost"
  ],
  "primary_tfm_models": [
    "tabiclv2",
    "tabpfn3"
  ],
  "boundary_models": [
    "catboost"
  ],
  "frozen_revisions": {
    "core_tfm_commit": "3b7e927dc1e47dc58e1ebd792820bc6f153caf88",
    "protocol_revision": "rtx3050_cuda_seed42_train256_test128",
    "execution_platform": "windows_local_nvidia_cuda",
    "bounded_context_protocol": {
      "max_train_rows": 256,
      "max_test_rows": 128
    },
    "hardware_target": "RTX 3050 Laptop 4GB"
  },
  "gpu_names": [
    "NVIDIA GeForce RTX 3050 Laptop GPU"
  ],
  "nvidia_smi": "NVIDIA GeForce RTX 3050 Laptop GPU, 4096 MiB, 546.92"
}


In [8]:
# Cell 11 — schema admission for every dataset plus one small real integration per model.
admission = {"datasets": {}, "real_model_integration": {}}

for dataset_name in DATASETS:
    try:
        X, a, b, provenance = prepared_dataset(dataset_name)
        splits, route = validated_outer_splits(X, a, b, OUTER_FOLDS)
        assert len(splits) == OUTER_FOLDS
        admission["datasets"][dataset_name] = {
            "ok": True, "rows": len(X), "features": X.shape[1],
            "stratification": route,
            "classes_a": int(a.nunique()), "classes_b": int(b.nunique()),
            "min_class_a": int(a.astype(str).value_counts().min()),
            "min_class_b": int(b.astype(str).value_counts().min()),
            "provenance": provenance,
        }
    except Exception as exc:
        admission["datasets"][dataset_name] = {
            "ok": False, "error": repr(exc), "traceback": traceback.format_exc(),
        }

X, a, b, _ = prepared_dataset("car")
splits, _ = validated_outer_splits(X, a, b, OUTER_FOLDS)
outer_train, outer_test = splits[0]
rng = np.random.default_rng(SEED)

# Grow the bounded integration subset until both target class sets are covered.
train_idx = None
for size in (96, 160, 256, min(512, len(outer_train)), len(outer_train)):
    size = min(size, len(outer_train))
    candidate = np.sort(rng.choice(outer_train, size=size, replace=False))
    if set(a.iloc[candidate].astype(str)) == set(a.astype(str)) and set(b.iloc[candidate].astype(str)) == set(b.astype(str)):
        train_idx = candidate
        break
assert train_idx is not None
seen_a, seen_b = set(a.iloc[train_idx].astype(str)), set(b.iloc[train_idx].astype(str))
eligible = np.array([
    i for i in outer_test
    if str(a.iloc[i]) in seen_a and str(b.iloc[i]) in seen_b
], dtype=int)
test_idx = eligible[:16]
assert len(test_idx) > 0

for model_name in MODELS:
    started = time.time()
    print(f"START real integration: car × {model_name}", flush=True)
    try:
        probe = extract(preflight_factory_for(model_name), X, a, b, train_idx, test_idx)
        for q in (probe.predictions.j_b_then_a, probe.predictions.j_a_then_b):
            assert np.isfinite(q).all()
            assert np.allclose(q.sum((1, 2)), 1, atol=1e-5)
        admission["real_model_integration"][model_name] = {
            "ok": True, "seconds": time.time() - started
        }
    except Exception as exc:
        admission["real_model_integration"][model_name] = {
            "ok": False, "seconds": time.time() - started,
            "error": repr(exc), "traceback": traceback.format_exc(),
        }
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

(RUN / "admission_preflight.json").write_text(
    json.dumps(admission, indent=2), encoding="utf-8"
)
dataset_ok = all(item["ok"] for item in admission["datasets"].values())
model_ok = all(item["ok"] for item in admission["real_model_integration"].values())
print(json.dumps(admission, indent=2))
assert dataset_ok and model_ok, "Admission preflight failed; inspect its JSON file."


START real integration: car × tabiclv2


START real integration: car × tabpfn3


START real integration: car × catboost


{
  "datasets": {
    "anneal": {
      "ok": true,
      "rows": 898,
      "features": 37,
      "stratification": "joint(A,B); seed=42; train-class coverage checked",
      "classes_a": 5,
      "classes_b": 8,
      "min_class_a": 8,
      "min_class_b": 10,
      "provenance": {
        "route": "OpenML through repository loader",
        "dataset": "anneal",
        "post_preprocessing_sha256": "b5fb6ed4b470778c12def10c5bdc353fc846e5fc1b45ece07172f9417d777c2d",
        "target_rows_removed_by_notebook": 0,
        "post_clean_rows": 898,
        "target_cleaning": "drop true missing, empty-string, and literal 'nan' targets"
      }
    },
    "car": {
      "ok": true,
      "rows": 1728,
      "features": 5,
      "stratification": "joint(A,B); seed=42; train-class coverage checked",
      "classes_a": 4,
      "classes_b": 3,
      "min_class_a": 65,
      "min_class_b": 576,
      "provenance": {
        "route": "OpenML through repository loader",
        "dataset": "car",
  

In [9]:
# Cell 12A — shared fast resumable runner + shard 1/5 (Anneal, Diamonds).
# Run this cell first. Dataset reconstruction is cached once per runtime.
if "RUN" not in globals() or "evaluate_fold" not in globals():
    raise RuntimeError(
        "Notebook state is not initialized. Run Cells 2–11 first, then rerun Cell 12A."
    )
PRIMARY_PATH = RUN / "fold_results.csv"
ABLATION_PATH = RUN / "selection_ablations.csv"
SENSITIVITY_PATH = RUN / "validation_fraction_sensitivity.csv"
FAILURE_PATH = RUN / "fold_failures.json"
MANIFEST_FOLD_PATH = RUN / "fold_manifests.json"
MIGRATION_MARKER = RUN / f"migration_{PROTOCOL_REVISION}.json"
INVALIDATED_DATASETS = {"mic", "nursery", "diamonds"}
SHARD_TIME_BUDGET_MINUTES = 120


def read_csv_or_empty(path):
    return pd.read_csv(path) if path.exists() and path.stat().st_size else pd.DataFrame()


def atomic_csv(frame, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def append_unique(existing, new_rows, keys):
    new_frame = pd.DataFrame(new_rows)
    if existing.empty:
        return new_frame.drop_duplicates(keys, keep="last")
    if new_frame.empty:
        return existing
    return pd.concat([existing, new_frame], ignore_index=True).drop_duplicates(
        keys, keep="last"
    )


# Preserve the Cell 6 implementation, but cache its result. This avoids repeated
# OpenML/UCI loading, Wine downloads, Diamonds reconstruction, and hashing.
if not getattr(prepared_dataset, "_core_tfm_cache_wrapper", False):
    _base_prepared_dataset = prepared_dataset
else:
    _base_prepared_dataset = globals()["_base_prepared_dataset"]
_prepared_dataset_cache = {}


def prepared_dataset(dataset_name):
    if dataset_name not in _prepared_dataset_cache:
        print(f"CACHE dataset once: {dataset_name}", flush=True)
        _prepared_dataset_cache[dataset_name] = _base_prepared_dataset(dataset_name)
    return _prepared_dataset_cache[dataset_name]


prepared_dataset._core_tfm_cache_wrapper = True


def migrate_legacy_results_once():
    if MIGRATION_MARKER.exists():
        return
    specifications = [
        (PRIMARY_PATH, "fold_results"),
        (ABLATION_PATH, "selection_ablations"),
        (SENSITIVITY_PATH, "validation_sensitivity"),
    ]
    for path, label in specifications:
        frame = read_csv_or_empty(path)
        if frame.empty or "model" not in frame.columns:
            continue
        invalid = (
            ~frame["model"].isin(MODELS)
            | frame["dataset"].isin(INVALIDATED_DATASETS)
        )
        legacy, active = frame[invalid].copy(), frame[~invalid].copy()
        if not legacy.empty:
            archive = RUN / f"excluded_legacy_{label}.csv"
            legacy.to_csv(archive, index=False)
            atomic_csv(active, path)
            print(f"Archived {len(legacy)} legacy rows to {archive.name}")

    manifests = (
        json.loads(MANIFEST_FOLD_PATH.read_text())
        if MANIFEST_FOLD_PATH.exists() else []
    )
    invalid_manifests = [
        item for item in manifests
        if item.get("model") not in MODELS
        or item.get("dataset") in INVALIDATED_DATASETS
    ]
    active_manifests = [item for item in manifests if item not in invalid_manifests]
    if invalid_manifests:
        (RUN / "excluded_legacy_fold_manifests.json").write_text(
            json.dumps(invalid_manifests, indent=2), encoding="utf-8"
        )
        MANIFEST_FOLD_PATH.write_text(
            json.dumps(active_manifests, indent=2), encoding="utf-8"
        )
    MIGRATION_MARKER.write_text(json.dumps({
        "protocol_revision": PROTOCOL_REVISION,
        "archived_model": "tabfm",
        "invalidated_and_rerun_datasets": sorted(INVALIDATED_DATASETS),
        "reason": "Corrected target construction and training-class coverage.",
    }, indent=2), encoding="utf-8")


def run_dataset_shard(shard_name, shard_datasets, time_budget_minutes=SHARD_TIME_BUDGET_MINUTES):
    """Run/resume only the requested datasets and checkpoint after every fold."""
    unknown = set(shard_datasets) - set(DATASETS)
    if unknown:
        raise KeyError(f"Unknown shard datasets: {sorted(unknown)}")

    folds = read_csv_or_empty(PRIMARY_PATH)
    folds = folds[folds["model"].isin(MODELS)].copy() if not folds.empty else folds
    ablations = read_csv_or_empty(ABLATION_PATH)
    ablations = (
        ablations[ablations["model"].isin(MODELS)].copy()
        if not ablations.empty else ablations
    )
    sensitivity = read_csv_or_empty(SENSITIVITY_PATH)
    sensitivity = (
        sensitivity[sensitivity["model"].isin(MODELS)].copy()
        if not sensitivity.empty else sensitivity
    )
    failures = json.loads(FAILURE_PATH.read_text()) if FAILURE_PATH.exists() else []
    manifests = (
        json.loads(MANIFEST_FOLD_PATH.read_text())
        if MANIFEST_FOLD_PATH.exists() else []
    )
    manifests = [item for item in manifests if item.get("model") in MODELS]

    completed = set()
    if not folds.empty:
        counts = folds.groupby(["dataset", "model", "fold"])["method"].nunique()
        completed = {tuple(key) for key, count in counts.items() if count >= 8}

    tasks = [
        (dataset_name, model_name, fold)
        for dataset_name in shard_datasets
        for model_name in MODELS
        for fold in range(1, OUTER_FOLDS + 1)
    ]
    remaining = [task for task in tasks if task not in completed]
    print(
        f"{shard_name}: {len(tasks) - len(remaining)}/{len(tasks)} complete; "
        f"{len(remaining)} remaining", flush=True,
    )

    session_started = time.time()
    for task_number, (dataset_name, model_name, fold) in enumerate(remaining, 1):
        if (time.time() - session_started) / 60 >= time_budget_minutes:
            print(f"{shard_name}: time budget reached safely; rerun this cell.")
            break
        print(
            f"START {shard_name} {task_number}/{len(remaining)}: "
            f"{dataset_name} × {model_name} fold {fold}", flush=True,
        )
        started = time.time()
        try:
            primary_rows, ablation_rows, sensitivity_rows, manifest = evaluate_fold(
                dataset_name, model_name, fold
            )
            folds = append_unique(
                folds, primary_rows, ["dataset", "model", "fold", "method"]
            )
            ablations = append_unique(
                ablations, ablation_rows,
                ["dataset", "model", "fold", "family"],
            )
            if sensitivity_rows:
                sensitivity = append_unique(
                    sensitivity, sensitivity_rows,
                    ["dataset", "model", "fold", "validation_fraction"],
                )
            manifest["seconds"] = time.time() - started
            manifests = [
                item for item in manifests
                if (item["dataset"], item["model"], item["fold"])
                != (dataset_name, model_name, fold)
            ] + [manifest]

            atomic_csv(folds, PRIMARY_PATH)
            atomic_csv(ablations, ABLATION_PATH)
            if not sensitivity.empty:
                atomic_csv(sensitivity, SENSITIVITY_PATH)
            MANIFEST_FOLD_PATH.write_text(
                json.dumps(manifests, indent=2), encoding="utf-8"
            )
            print(
                f"PASS {dataset_name} × {model_name} fold {fold} "
                f"({time.time() - started:.1f}s)", flush=True,
            )
        except Exception as exc:
            failures.append({
                "dataset": dataset_name, "model": model_name, "fold": fold,
                "shard": shard_name, "protocol_revision": PROTOCOL_REVISION,
                "error": repr(exc), "traceback": traceback.format_exc(),
                "created_utc": time.strftime(
                    "%Y-%m-%dT%H:%M:%SZ", time.gmtime()
                ),
            })
            FAILURE_PATH.write_text(
                json.dumps(failures, indent=2), encoding="utf-8"
            )
            print("FAILED:", repr(exc), flush=True)
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    current = read_csv_or_empty(PRIMARY_PATH)
    current = (
        current[current["model"].isin(MODELS)].copy()
        if not current.empty else current
    )
    counts = (
        current.groupby(["dataset", "model", "fold"])["method"].nunique()
        if not current.empty else pd.Series(dtype=int)
    )
    shard_keys = {
        (dataset_name, model_name, fold)
        for dataset_name in shard_datasets
        for model_name in MODELS
        for fold in range(1, OUTER_FOLDS + 1)
    }
    completed_keys = {tuple(key) for key, count in counts.items() if count >= 8}
    shard_complete = len(shard_keys & completed_keys)
    global_required = len(DATASETS) * len(MODELS) * OUTER_FOLDS
    print(
        f"{shard_name}: persistent progress {shard_complete}/{len(shard_keys)}; "
        f"global {len(completed_keys)}/{global_required}", flush=True,
    )

    (RUN / "manifest.json").write_text(json.dumps({
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "python": sys.version, "platform": platform.platform(),
        "config": protocol, "fold_manifests": manifests,
        "failure_attempts": failures,
    }, indent=2), encoding="utf-8")
    return shard_complete == len(shard_keys)


migrate_legacy_results_once()
SHARD_1_COMPLETE = run_dataset_shard("SHARD 1/5", ["anneal", "diamonds"])


SHARD 1/5: 30/30 complete; 0 remaining


SHARD 1/5: persistent progress 30/30; global 104/150


In [10]:
# Cell 12B — shard 2/5 (Car, Nursery). Rerun only this cell if it times out.
SHARD_2_COMPLETE = run_dataset_shard("SHARD 2/5", ["car", "nursery"])


SHARD 2/5: 30/30 complete; 0 remaining


SHARD 2/5: persistent progress 30/30; global 104/150


In [11]:
# Cell 12C — shard 3/5 (Wine, Credit). Rerun only this cell if it times out.
SHARD_3_COMPLETE = run_dataset_shard("SHARD 3/5", ["wine", "credit"])


SHARD 3/5: 30/30 complete; 0 remaining


SHARD 3/5: persistent progress 30/30; global 104/150


In [12]:
# Cell 12D — shard 4/5 (MIC, Customer). Rerun only this cell if it times out.
SHARD_4_COMPLETE = run_dataset_shard("SHARD 4/5", ["mic", "customer"])


SHARD 4/5: 14/30 complete; 16 remaining


START SHARD 4/5 1/16: mic × catboost fold 5


CACHE dataset once: mic


PASS mic × catboost fold 5 (219.3s)


START SHARD 4/5 2/16: customer × tabiclv2 fold 1


...

START SHARD 4/5 15/16: customer × catboost fold 4


PASS customer × catboost fold 4 (54.9s)


START SHARD 4/5 16/16: customer × catboost fold 5


PASS customer × catboost fold 5 (53.6s)


SHARD 4/5: persistent progress 30/30; global 120/150


In [13]:
# Cell 12E — shard 5/5 (Marketing, Phishing) and global completion report.
SHARD_5_COMPLETE = run_dataset_shard("SHARD 5/5", ["marketing", "phishing"])

folds = read_csv_or_empty(PRIMARY_PATH)
folds = folds[folds["model"].isin(MODELS)].copy()
completed = folds.groupby(["dataset", "model", "fold"])["method"].nunique()
required = len(DATASETS) * len(MODELS) * OUTER_FOLDS
actual = int((completed >= 8).sum())
print(f"ACTIVE MATRIX: {actual}/{required} folds complete.")
if actual == required:
    print("ACTIVE MATRIX COMPLETE. Continue to the next cell.")
else:
    print("Rerun whichever shard reports incomplete; completed folds will be skipped.")


SHARD 5/5: 0/30 complete; 30 remaining


START SHARD 5/5 1/30: marketing × tabiclv2 fold 1


CACHE dataset once: marketing


PASS marketing × tabiclv2 fold 1 (14.7s)


START SHARD 5/5 2/30: marketing × tabiclv2 fold 2


...

PASS phishing × catboost fold 4 (46.9s)


START SHARD 5/5 30/30: phishing × catboost fold 5


PASS phishing × catboost fold 5 (45.7s)


SHARD 5/5: persistent progress 30/30; global 150/150


ACTIVE MATRIX: 150/150 folds complete.
ACTIVE MATRIX COMPLETE. Continue to the next cell.
